# 05 — Spillover Regression

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 1******6

**Purpose:** Estimate whether US implied volatility smile parameters on day $t$ have predictive power for EU smile parameters on day $t+1$, using OLS with Newey-West heteroskedasticity- and autocorrelation-consistent (HAC) standard errors.

**Input:** `data/analysis_ready/panel_merged__<timestamp>.csv` (output of notebook 04)

**Outputs:**
- `data/outputs/regression_results__<timestamp>.csv` — coefficient table, all models
- `data/outputs/figures/` — smile parameter time series, residual diagnostics
- `logs/05_regression_log__<timestamp>.json`

---

## Regression specification

**Primary maturity node:** 30-day.  
Justification: highest liquidity in both equity option markets, primary node in Chen et al. (2022), and the maturity at which VIX is constructed. All other nodes (60, 91, 182, 365-day) are estimated as robustness checks.

**Three models — one per smile parameter:**

$$\text{ATM}^{EU}_{t+1} = \alpha + \beta_1 \text{ATM}^{US}_{t} + \beta_2 \Delta\text{VIX}_t + \beta_3 \text{ATM}^{EU}_{t} + \varepsilon_{t+1}$$

$$\text{Skew}^{EU}_{t+1} = \alpha + \beta_1 \text{Skew}^{US}_{t} + \beta_2 \Delta\text{VIX}_t + \beta_3 \text{Skew}^{EU}_{t} + \varepsilon_{t+1}$$

$$\text{Curvature}^{EU}_{t+1} = \alpha + \beta_1 \text{Curvature}^{US}_{t} + \beta_2 \Delta\text{VIX}_t + \beta_3 \text{Curvature}^{EU}_{t} + \varepsilon_{t+1}$$

**Regressors in each model:**
- $\text{Parameter}^{US}_t$ — same smile parameter from the US market on day $t$ (main variable of interest)
- $\Delta\text{VIX}_t = \text{VIX}_t - \text{VIX}_{t-1}$ — change in VIX (controls for global volatility shocks)
- $\text{Parameter}^{EU}_t$ — lagged EU parameter (controls for own-market persistence / autocorrelation)

**Standard errors:** Newey-West HAC with automatic lag selection (Newey and West, 1994).  
Justification: smile parameters are persistent time series; OLS standard errors will be biased downward. HAC correction is standard in this literature (Tompkins, 2001; Chen et al., 2022).

**Hypothesis of interest:** $\beta_1 > 0$ and statistically significant — US smile parameter predicts EU smile parameter after controlling for own-market persistence and global volatility.

---

## References
- Chen, J., Han, Q., Ryu, D. and Tang, J. (2022). Does the world smile together? *Journal of International Financial Markets, Institutions and Money*, 77, 101497.
- Malz, A.M. (1997). Estimating the probability distribution of the future exchange rate from option prices. *Journal of Derivatives*, 5(2), 18–36.
- Newey, W.K. and West, K.D. (1994). Automatic lag selection in covariance matrix estimation. *Review of Economic Studies*, 61(4), 631–653.
- Tompkins, R.G. (2001). Implied volatility surfaces: Uncovering regularities for options on financial futures. *European Journal of Finance*, 7(3), 198–230.

---
## Step 0 — Imports and configuration

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import json
import warnings
from datetime import datetime

import subprocess
subprocess.run(["pip", "install", "statsmodels"])
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller

import matplotlib
matplotlib.use('Agg')  # non-interactive backend — safe for Jupyter
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore', category=FutureWarning)

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

ANALYSIS_READY_DIR = '../data/analysis_ready/'
OUTPUT_DIR         = '../data/outputs/'
FIGURES_DIR        = '../data/outputs/figures/'
LOG_DIR            = '../logs/'

for d in [OUTPUT_DIR, FIGURES_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# Primary maturity node
PRIMARY_NODE = 30

# Robustness nodes (estimated after primary)
ROBUSTNESS_NODES = [60, 91, 182, 365]

# Smile parameters to model
PARAMETERS = ['atm_iv', 'skew', 'curvature']

# Newey-West lag selection: use int(4 * (T/100)^(2/9)) — standard rule of thumb
# Will be computed per regression once T is known

log = {
    'notebook': '05_regression',
    'run_timestamp': RUN_TIMESTAMP,
    'primary_node': PRIMARY_NODE,
    'robustness_nodes': ROBUSTNESS_NODES,
    'parameters': PARAMETERS,
    'steps': {}
}

print(f'Run timestamp: {RUN_TIMESTAMP}')
print(f'Primary maturity node: {PRIMARY_NODE}-day')
print(f'Robustness nodes: {ROBUSTNESS_NODES}-day')

Matplotlib is building the font cache; this may take a moment.


Run timestamp: 20260314_160035
Primary maturity node: 30-day
Robustness nodes: [60, 91, 182, 365]-day


---
## Step 1 — Load merged panel

In [10]:
pattern = os.path.join(ANALYSIS_READY_DIR, 'panel_merged__*.csv')
matches = glob.glob(pattern)

if len(matches) == 0:
    raise FileNotFoundError(
        f'No merged panel found at {pattern}. '
        'Run notebook 04 first.'
    )

latest = max(matches, key=os.path.getmtime)
panel = pd.read_csv(latest, parse_dates=['date_eu'])

print(f'Loaded: {latest}')
print(f'Shape: {panel.shape}')
print(f'Columns: {panel.columns.tolist()}')
print()

if len(panel) == 0:
    print('WARNING: Merged panel has zero rows — EU and US windows do not overlap.')
    print('Running pipeline validation mode: within-market autocorrelation on EU panel only.')
    print()
    # Load EU intermediate directly for pipeline validation
    import glob as _glob
    eu_files = sorted(_glob.glob('../data/intermediate/eu2013_analysis_ready__*.csv'))
    if not eu_files:
        raise FileNotFoundError('No EU intermediate file found. Run notebook 03a first.')
    panel = pd.read_csv(eu_files[-1], parse_dates=['date'])
    panel = panel.rename(columns={'date': 'date_eu'})
    # Create lagged versions of all smile columns as pseudo-US regressors
    for col in ['atm_iv', 'skew', 'curvature']:
        panel[f'eu_{col}'] = panel[col]
        panel[f'us_{col}'] = panel.groupby('days')[col].shift(1)
    panel['us_vix'] = panel['vix']
    panel['eu_vix'] = panel['vix']
    panel = panel.dropna(subset=['us_atm_iv', 'us_skew', 'us_curvature'])
    panel = panel.reset_index(drop=True)
    print(f'Validation panel shape: {panel.shape}')
    print(f'Dates: {panel["date_eu"].min().date()} to {panel["date_eu"].max().date()}')
    print()
    print('NOTE: These results are pipeline validation only.')
    print('EU t predicting EU t+1 — NOT a cross-market spillover result.')
    print('Do not interpret as thesis findings.')

print(f'Date range: {panel["date_eu"].min().date()} to {panel["date_eu"].max().date()}')
print(f'Unique EU dates: {panel["date_eu"].nunique()}')
print(f'Maturity nodes present: {sorted(panel["days"].unique().tolist())}')

log['steps']['step1_load'] = {
    'file': latest,
    'shape': list(panel.shape),
    'date_min': str(panel['date_eu'].min().date()),
    'date_max': str(panel['date_eu'].max().date()),
    'n_dates': int(panel['date_eu'].nunique()),
    'nodes': sorted(panel['days'].unique().tolist()),
}

Loaded: ../data/analysis_ready/panel_merged__20260314_155525.csv
Shape: (0, 40)
Columns: ['date_eu', 'days', 'eu_atm_iv', 'eu_skew', 'eu_curvature', 'eu_put25_iv', 'eu_call75_iv', 'eu_us_rf_rate', 'eu_us_yield_1y', 'eu_us_yield_10y', 'eu_us_term_spread', 'eu_us_hy_spread', 'eu_sp500_ret', 'eu_sp500_idx', 'eu_ff_mktrf', 'eu_ff_smb', 'eu_ff_hml', 'eu_ff_rf', 'eu_vix', 'eu_eurusd', 'eu_hist_vol_30d', 'date', 'us_atm_iv', 'us_skew', 'us_curvature', 'us_put25_iv', 'us_call75_iv', 'us_us_rf_rate', 'us_us_yield_1y', 'us_us_yield_10y', 'us_us_term_spread', 'us_us_hy_spread', 'us_sp500_ret', 'us_sp500_idx', 'us_ff_mktrf', 'us_ff_smb', 'us_ff_hml', 'us_ff_rf', 'us_vix', 'us_eurusd']

Running pipeline validation mode: within-market autocorrelation on EU panel only.

Validation panel shape: (100, 29)
Dates: 2013-03-04 to 2013-03-15

NOTE: These results are pipeline validation only.
EU t predicting EU t+1 — NOT a cross-market spillover result.
Do not interpret as thesis findings.
Date range: 2013-0

---
## Step 2 — Construct regression variables

Build the full variable set needed for all three models:
- ΔVIX: first difference of VIX (controls for global volatility shocks)
- Lagged EU parameters: own-market persistence control
- All constructed per maturity node

In [11]:
def build_regression_df(panel, node):
    """
    Given the full merged panel and a maturity node (days),
    returns a clean regression DataFrame for that node.

    Columns returned:
      date_eu           — EU observation date (dependent variable date = t+1)
      eu_atm_iv         — EU ATM IV at t+1  (dependent variable)
      eu_skew           — EU Skew at t+1    (dependent variable)
      eu_curvature      — EU Curvature at+1 (dependent variable)
      us_atm_iv         — US ATM IV at t     (main regressor)
      us_skew           — US Skew at t       (main regressor)
      us_curvature      — US Curvature at t  (main regressor)
      delta_vix         — ΔVIX at t          (control)
      eu_atm_iv_lag1    — EU ATM IV at t     (own-market persistence control)
      eu_skew_lag1      — EU Skew at t       (own-market persistence control)
      eu_curvature_lag1 — EU Curvature at t  (own-market persistence control)

    The lag-1 EU variables are constructed by shifting eu_ columns by 1
    within the node-specific slice, sorted by date.
    """
    df = panel[panel['days'] == node].copy()
    df = df.sort_values('date_eu').reset_index(drop=True)

    # ΔVIX: use eu_vix which is the EU-side VIX (same global series, just the
    # value on date t+1). We need VIX at t, which is us_vix in the merged panel.
    # ΔVIX_t = us_vix_t - us_vix_{t-1}
    df['delta_vix'] = df['us_vix'].diff()

    # Own-market persistence: EU parameter at t = lag-1 of EU parameter at t+1
    # Since panel is sorted by date_eu (= t+1), shift(1) gives t
    for param in PARAMETERS:
        df[f'eu_{param}_lag1'] = df[f'eu_{param}'].shift(1)

    # Drop first row (NaN from diff and shift)
    df = df.dropna(subset=['delta_vix'] + [f'eu_{p}_lag1' for p in PARAMETERS])
    df = df.reset_index(drop=True)

    print(f'Node {node}-day: {len(df)} observations after lag construction')
    return df


# Build for primary node
reg_primary = build_regression_df(panel, PRIMARY_NODE)

# Build for robustness nodes
reg_robustness = {}
for node in ROBUSTNESS_NODES:
    if node in panel['days'].unique():
        reg_robustness[node] = build_regression_df(panel, node)
    else:
        print(f'Node {node}-day not present in merged panel — skipping.')

print()
print(f'Primary regression dataset shape: {reg_primary.shape}')
print()
print('Sample (first 3 rows):')
display_cols = ['date_eu', 'eu_atm_iv', 'eu_skew', 'eu_curvature',
                'us_atm_iv', 'us_skew', 'us_curvature', 'delta_vix']
available = [c for c in display_cols if c in reg_primary.columns]
print(reg_primary[available].head(3).to_string())

log['steps']['step2_variables'] = {
    'primary_node_obs': len(reg_primary),
    'robustness_nodes_obs': {str(k): len(v) for k, v in reg_robustness.items()},
}

Node 30-day: 9 observations after lag construction
Node 60-day: 9 observations after lag construction
Node 91-day: 9 observations after lag construction
Node 182-day: 9 observations after lag construction
Node 365-day: 9 observations after lag construction

Primary regression dataset shape: (9, 33)

Sample (first 3 rows):
     date_eu  eu_atm_iv   eu_skew  eu_curvature  us_atm_iv   us_skew  us_curvature  delta_vix
0 2013-03-05   0.218637 -0.001658      0.030119   0.227141 -0.000624      0.030300      -0.53
1 2013-03-06   0.224246 -0.000110      0.027536   0.218637 -0.001658      0.030119       0.05
2 2013-03-07   0.199135 -0.000691      0.015534   0.224246 -0.000110      0.027536      -0.47


---
## Step 3 — Pre-regression diagnostics

Before estimating, run two standard checks:

1. **ADF unit root test** on each dependent variable — if a series has a unit root, OLS on levels is spurious. We test and report; if unit root is found, we switch to first differences for that variable and flag it.
2. **Summary statistics** for all regression variables.

In [12]:
print('=== PRE-REGRESSION DIAGNOSTICS (PRIMARY NODE: 30-day) ===')
print()

# Summary statistics
reg_cols = (
    [f'eu_{p}' for p in PARAMETERS] +
    [f'us_{p}' for p in PARAMETERS] +
    ['delta_vix'] +
    [f'eu_{p}_lag1' for p in PARAMETERS]
)
available_reg_cols = [c for c in reg_cols if c in reg_primary.columns]

print('Summary statistics:')
print(reg_primary[available_reg_cols].describe().round(6).to_string())
print()

# ADF unit root tests on dependent variables
# H0: series has a unit root (non-stationary)
# If p-value > 0.05: cannot reject unit root → flag for differencing
print('ADF unit root tests (H0: unit root present):')
print(f'{"Variable":<25} {"ADF stat":>10} {"p-value":>10} {"Decision":>20}')
print('-' * 70)

adf_results = {}
unit_root_flags = []

for param in PARAMETERS:
    col = f'eu_{param}'
    if col not in reg_primary.columns:
        continue
    series = reg_primary[col].dropna()
    if len(series) < 10:
        print(f'{col:<25} {"N/A":>10} {"N/A":>10} {"Too few obs":>20}')
        continue
    result = adfuller(series, autolag='AIC')
    adf_stat = result[0]
    p_val    = result[1]
    decision = 'Stationary ✓' if p_val <= 0.05 else 'Unit root — FLAG'
    if p_val > 0.05:
        unit_root_flags.append(col)
    adf_results[col] = {'adf_stat': round(adf_stat, 4), 'p_value': round(p_val, 4)}
    print(f'{col:<25} {adf_stat:>10.4f} {p_val:>10.4f} {decision:>20}')

print()
if unit_root_flags:
    print(f'Unit root flagged in: {unit_root_flags}')
    print('These variables will be first-differenced in their respective regressions.')
    print('Results for both levels and differences will be reported for transparency.')
else:
    print('No unit roots detected. Proceeding with levels.')

log['steps']['step3_diagnostics'] = {
    'adf_results': adf_results,
    'unit_root_flags': unit_root_flags,
}

=== PRE-REGRESSION DIAGNOSTICS (PRIMARY NODE: 30-day) ===

Summary statistics:
       eu_atm_iv   eu_skew  eu_curvature  us_atm_iv   us_skew  us_curvature  delta_vix  eu_atm_iv_lag1  eu_skew_lag1  eu_curvature_lag1
count   9.000000  9.000000      9.000000   9.000000  9.000000      9.000000   9.000000        9.000000      9.000000           9.000000
mean    0.201940 -0.000608      0.021664   0.205493 -0.000646      0.022755  -0.301111        0.205493     -0.000646           0.022755
std     0.011239  0.000577      0.007230   0.013629  0.000564      0.007751   0.493544        0.013629      0.000564           0.007751
min     0.194199 -0.001658      0.010744   0.194199 -0.001658      0.010744  -1.030000        0.194199     -0.001658           0.010744
25%     0.195671 -0.000929      0.016118   0.196186 -0.000929      0.016118  -0.530000        0.196186     -0.000929           0.016118
50%     0.196248 -0.000652      0.020483   0.197977 -0.000652      0.022686  -0.470000        0.197977   

## Step 4 — Estimate primary regressions (30-day node)

Three separate OLS regressions, one per smile parameter.  
Standard errors: Newey-West HAC, lag = $\lfloor 4(T/100)^{2/9} \rfloor$ (Newey and West, 1994 automatic rule for the Bartlett kernel, Table II Panel C line 6).

**Implementation note:** The formula $\lfloor 4(T/100)^{2/9} \rfloor$ is the *lag selection parameter* n from Newey and West (1994), which controls the rate at which the optimal bandwidth grows with sample size. Following standard applied practice, n is passed directly as the HAC bandwidth (maxlags), rather than implementing the full two-step data-dependent bandwidth estimator. This simplification is conventional in the empirical finance literature and is explicitly noted here for transparency. The Bartlett kernel is used throughout, consistent with the paper's recommendation and statsmodels' default.

Each model is:

$$Y^{EU}_{t+1} = \alpha + \beta_1 Y^{US}_t + \beta_2 \Delta\text{VIX}_t + \beta_3 Y^{EU}_t + \varepsilon_{t+1}$$

where $Y$ is ATM IV, Skew, or Curvature, respectively.

**Regressors:**
- $Y^{US}_t$: US smile parameter on day $t$ — main variable of interest; $\hat{\beta}_1 > 0$ and significant is consistent with the spillover hypothesis
- $\Delta\text{VIX}_t = \text{VIX}_t - \text{VIX}_{t-1}$: controls for global volatility shocks common to both markets
- $Y^{EU}_t$: lagged own-market parameter; controls for persistence in the EU smile series

**OLS point estimates are unaffected by the HAC correction — only standard errors change.**  
Regressions are skipped if $T < 10$ (insufficient degrees of freedom for HAC estimation to be meaningful).

In [18]:
def newey_west_lags(T):
    """
    Newey-West (1994) automatic lag selection rule for the Bartlett kernel.

    Source: Newey & West (1994), 'Automatic Lag Selection in Covariance Matrix
    Estimation', Review of Economic Studies, 61(4), 631-653. Table II, Panel C,
    line 6 (Bartlett kernel, no prewhitening).

    Formula: n = floor(4 * (T/100)^(2/9))

    This is the lag selection parameter n, which governs the rate at which the
    optimal HAC bandwidth grows with sample size. Following standard applied
    practice, n is passed directly as the HAC bandwidth (maxlags) rather than
    implementing the full two-step data-dependent bandwidth estimator. This
    simplification is conventional in empirical finance (e.g. Chen et al., 2022;
    Tompkins, 2001) and is documented in the Step 4 markdown cell.

    Parameters
    ----------
    T : int
        Number of usable observations in the regression sample.

    Returns
    -------
    int
        Number of lags to pass as maxlags to the HAC estimator.
    """
    return int(np.floor(4 * (T / 100) ** (2 / 9)))


def run_ols_nw(df, dep_var, indep_vars, label):
    """
    Run OLS with Newey-West HAC standard errors (Bartlett kernel).

    OLS point estimates (beta-hat) are unaffected by the HAC correction.
    Only standard errors, t-statistics, and p-values use the HAC covariance
    matrix. The Bartlett kernel is used throughout, consistent with Newey &
    West (1994) and statsmodels' default for cov_type='HAC'.

    Parameters
    ----------
    df         : pd.DataFrame
                 Must contain dep_var, all indep_vars, and no structural breaks
                 in the index (sorted by date before passing in).
    dep_var    : str
                 Name of the dependent variable column (Y^EU_{t+1}).
    indep_vars : list of str
                 Names of independent variable columns. Expected order:
                 [us_<param>, delta_vix, eu_<param>_lag1]. Missing columns
                 are handled upstream by indep_avail filtering.
    label      : str
                 Human-readable label for printed output and log.

    Returns
    -------
    results      : statsmodels RegressionResultsWrapper, or None if skipped.
    summary_dict : dict of key results for logging, or None if skipped.

    Skip condition
    --------------
    T < 10: with 4 regressors (const + 3), fewer than 10 observations leaves
    fewer than 6 degrees of freedom. HAC estimation is not meaningful and
    statsmodels may raise warnings or produce degenerate results.
    """
    # --- 1. Drop NaNs in variables used for this regression only -----------
    cols_needed = [dep_var] + indep_vars
    clean = df[cols_needed].dropna()
    T = len(clean)

    if T < 10:
        print(f'{label}: insufficient observations (T={T}). Skipping.')
        return None, None

    # --- 2. Construct y and X ----------------------------------------------
    y = clean[dep_var]
    X = sm.add_constant(clean[indep_vars])
    # sm.add_constant adds an intercept column named 'const'.
    # If all values of a regressor are identical (no variation), OLS will
    # raise a perfect multicollinearity warning — acceptable to surface.

    # --- 3. OLS fit with Newey-West HAC covariance -------------------------
    # cov_type='HAC' uses the Bartlett kernel by default in statsmodels.
    # maxlags = n from Newey & West (1994), used directly as the bandwidth.
    # use_correction=False matches the standard finite-sample formula.
    model   = sm.OLS(y, X)
    nw_lags = newey_west_lags(T)
    results = model.fit(
        cov_type='HAC',
        cov_kwds={'maxlags': nw_lags, 'use_correction': False}
    )

    # --- 4. Residual autocorrelation diagnostic ----------------------------
    # Durbin-Watson: 2.0 = no autocorrelation; <1.5 or >2.5 = concern.
    # Note: DW is computed on OLS residuals (not HAC-adjusted). It diagnoses
    # whether residual autocorrelation remains after the HAC correction is
    # applied to SEs. A value far from 2.0 does not invalidate HAC SEs but
    # suggests the lag truncation may be too short.
    dw_stat = durbin_watson(results.resid)

    # --- 5. Print formatted output -----------------------------------------
    print(f'--- {label} ---')
    print(f'T={T}  |  NW lags (n)={nw_lags}  |  Kernel=Bartlett')
    print(f'R-squared: {results.rsquared:.4f}   Adj R-squared: {results.rsquared_adj:.4f}')
    print(f'Durbin-Watson: {dw_stat:.4f}  (2.0 = no residual autocorrelation)')
    print()
    print(f'{"Variable":<25} {"Coef":>10} {"NW SE":>10} {"t-stat":>10} {"p-value":>10} {"":>5}')
    print('-' * 75)
    for var in results.params.index:
        coef  = results.params[var]
        se    = results.bse[var]
        tstat = results.tvalues[var]
        pval  = results.pvalues[var]
        stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f'{var:<25} {coef:>10.6f} {se:>10.6f} {tstat:>10.4f} {pval:>10.4f} {stars:>5}')
    print()

    # --- 6. Build summary dict for logging ---------------------------------
    summary_dict = {
        'label':          label,
        'dep_var':        dep_var,
        'indep_vars':     indep_vars,
        'T':              T,
        'nw_lags':        nw_lags,
        'kernel':         'Bartlett',
        'r_squared':      round(results.rsquared, 6),
        'adj_r_squared':  round(results.rsquared_adj, 6),
        'durbin_watson':  round(dw_stat, 4),
        'coefficients': {
            v: {
                'coef':   round(results.params[v], 8),
                'se_nw':  round(results.bse[v], 8),
                'tstat':  round(results.tvalues[v], 4),
                'pvalue': round(results.pvalues[v], 6),
            } for v in results.params.index
        }
    }

    return results, summary_dict


# ---------------------------------------------------------------------------
# Run three primary regressions — 30-day node
# ---------------------------------------------------------------------------
print('====== PRIMARY REGRESSIONS — 30-day node ======')
print()
print('Specification: Y^EU_{t+1} = alpha + b1*Y^US_t + b2*dVIX_t + b3*Y^EU_t + e_{t+1}')
print('Standard errors: Newey-West HAC, Bartlett kernel, n = floor(4*(T/100)^(2/9))')
print('Source: Newey & West (1994), Review of Economic Studies, 61(4), Table II Panel C line 6')
print()

primary_results   = {}
primary_summaries = {}

for param in PARAMETERS:
    dep         = f'eu_{param}'
    indep       = [f'us_{param}', 'delta_vix', f'eu_{param}_lag1']
    indep_avail = [c for c in indep if c in reg_primary.columns]
    label       = f'Model: EU {param} (t+1) ~ US {param} (t) + delta_vix + EU {param} (t)'

    res, summ = run_ols_nw(reg_primary, dep, indep_avail, label)

    primary_results[param]   = res
    primary_summaries[param] = summ

log['steps']['step4_primary_regressions'] = {
    'node':              PRIMARY_NODE,
    'specification':     'Y_EU_{t+1} = a + b1*Y_US_t + b2*dVIX_t + b3*Y_EU_t + e',
    'se_method':         'Newey-West HAC, Bartlett kernel',
    'nw_formula':        'floor(4*(T/100)^(2/9))',
    'nw_source':         'Newey & West (1994), RES 61(4), Table II Panel C line 6',
    'use_correction':    False,
    'results':           primary_summaries,
}

====== PRIMARY REGRESSIONS — 30-day node ======

Specification: Y^EU_{t+1} = alpha + b1*Y^US_t + b2*dVIX_t + b3*Y^EU_t + e_{t+1}
Standard errors: Newey-West HAC, Bartlett kernel, n = floor(4*(T/100)^(2/9))
Source: Newey & West (1994), Review of Economic Studies, 61(4), Table II Panel C line 6

Model: EU atm_iv (t+1) ~ US atm_iv (t) + delta_vix + EU atm_iv (t): insufficient observations (T=9). Skipping.
Model: EU skew (t+1) ~ US skew (t) + delta_vix + EU skew (t): insufficient observations (T=9). Skipping.
Model: EU curvature (t+1) ~ US curvature (t) + delta_vix + EU curvature (t): insufficient observations (T=9). Skipping.


## Step 5 — Robustness checks: other maturity nodes

Repeat the same three models for each robustness node (60, 91, 182, 365-day) using the identical specification and Newey-West HAC standard errors as Step 4.

The primary conclusion (spillover in the 30-day node) should be robust to maturity choice if the spillover is a genuine cross-market phenomenon and not an artifact of the specific contract. Coefficient stability across nodes — in sign, magnitude, and significance — is the robustness criterion.

Same standard error convention: Bartlett kernel, $n = \lfloor 4(T/100)^{2/9} \rfloor$, computed separately per node using that node's own $T$.

In [19]:
# Step 5 — Robustness checks: other maturity nodes
# Identical specification to Step 4. run_ols_nw and newey_west_lags are
# reused without modification. NW lags are recomputed per node since T may differ across nodes.

robustness_log = {}

for node, df_node in reg_robustness.items():
    print(f'====== ROBUSTNESS — {node}-day node ======')
    print()

    node_summaries = {}

    for param in PARAMETERS:
        dep         = f'eu_{param}'
        indep       = [f'us_{param}', 'delta_vix', f'eu_{param}_lag1']
        indep_avail = [c for c in indep if c in df_node.columns]
        label       = f'{node}-day | EU {param} (t+1) ~ US {param} (t) + delta_vix + EU {param} (t)'

        _, summ = run_ols_nw(df_node, dep, indep_avail, label)
        node_summaries[param] = summ

    robustness_log[str(node)] = node_summaries

log['steps']['step5_robustness'] = {
    'nodes':         [str(n) for n in reg_robustness.keys()],
    'specification': 'Y_EU_{t+1} = a + b1*Y_US_t + b2*dVIX_t + b3*Y_EU_t + e',
    'se_method':     'Newey-West HAC, Bartlett kernel',
    'nw_formula':    'floor(4*(T/100)^(2/9)) — recomputed per node',
    'nw_source':     'Newey & West (1994), RES 61(4), Table II Panel C line 6',
    'results':       robustness_log,
}

====== ROBUSTNESS — 60-day node ======

60-day | EU atm_iv (t+1) ~ US atm_iv (t) + delta_vix + EU atm_iv (t): insufficient observations (T=9). Skipping.
60-day | EU skew (t+1) ~ US skew (t) + delta_vix + EU skew (t): insufficient observations (T=9). Skipping.
60-day | EU curvature (t+1) ~ US curvature (t) + delta_vix + EU curvature (t): insufficient observations (T=9). Skipping.
====== ROBUSTNESS — 91-day node ======

91-day | EU atm_iv (t+1) ~ US atm_iv (t) + delta_vix + EU atm_iv (t): insufficient observations (T=9). Skipping.
91-day | EU skew (t+1) ~ US skew (t) + delta_vix + EU skew (t): insufficient observations (T=9). Skipping.
91-day | EU curvature (t+1) ~ US curvature (t) + delta_vix + EU curvature (t): insufficient observations (T=9). Skipping.
====== ROBUSTNESS — 182-day node ======

182-day | EU atm_iv (t+1) ~ US atm_iv (t) + delta_vix + EU atm_iv (t): insufficient observations (T=9). Skipping.
182-day | EU skew (t+1) ~ US skew (t) + delta_vix + EU skew (t): insufficient obs

## Step 6 — Residual diagnostics (primary regressions only)

For each primary regression (30-day node), plot:

1. **Residuals over time**: should show no systematic pattern or trending structure; visible clustering or drift suggests remaining autocorrelation
2. **Residual histogram**: should be approximately symmetric and unimodal; severe skew or heavy tails are noted but not fatal (CLT applies asymptotically for large T)

**What each plot tells and what action it implies:**

- Residuals over time with clear pattern → autocorrelation remains in residuals despite HAC correction → consider increasing NW lag length as a robustness check (try 1.5× and 2× the automatic n)
- Durbin-Watson far from 2.0 (already reported in Step 4) is the quantitative companion to the visual residuals plot
- Non-normality in the histogram → does not invalidate OLS or HAC SEs asymptotically, but note it as a limitation if T is small
- These plots are diagnostic only — they do not change the reported coefficients or standard errors

In [22]:
# ---------------------------------------------------------------------------
# Step 6 — Residual diagnostics: primary regressions (30-day node)
# Two plots per model: residuals over time + residual histogram.
# Diagnostic only — nothing in this step modifies any regression output.
# ---------------------------------------------------------------------------

diag_figures = []

for param in PARAMETERS:
    res = primary_results.get(param)

    if res is None:
        print(f'No result for {param} — skipping diagnostic plot.')
        continue

    # Reconstruct the clean sample used in this regression
    # (must match exactly what run_ols_nw used, including dropna)
    dep         = f'eu_{param}'
    indep       = [f'us_{param}', 'delta_vix', f'eu_{param}_lag1']
    indep_avail = [c for c in indep if c in reg_primary.columns]
    clean       = reg_primary[[dep] + indep_avail].dropna().reset_index(drop=True)

    # Sanity check: residual count must match clean row count
    assert len(res.resid) == len(clean), (
        f'Residual length mismatch for {param}: '
        f'{len(res.resid)} residuals vs {len(clean)} clean obs. '
        'Check that run_ols_nw and this cell use identical dropna logic.'
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(
        f'Residual diagnostics — {param.upper()} model (30-day node)',
        fontsize=11, fontweight='bold'
    )

    # --- Plot 1: Residuals over time ---------------------------------------
    # X-axis: observation index (integer, since dates may not be unique
    # across nodes in the validation panel).
    axes[0].plot(clean.index, res.resid, color='steelblue', linewidth=0.9)
    axes[0].axhline(0, color='black', linewidth=0.7, linestyle='--')
    axes[0].set_title('Residuals over time')
    axes[0].set_xlabel('Observation index')
    axes[0].set_ylabel('Residual')
    axes[0].grid(True, alpha=0.3)
    # Flag: if residuals show clear trend or clustering, note in thesis.

    # --- Plot 2: Residual histogram ----------------------------------------
    axes[1].hist(res.resid, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='black', linewidth=0.7, linestyle='--')
    axes[1].set_title('Residual distribution')
    axes[1].set_xlabel('Residual')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    fig_path = os.path.join(
        FIGURES_DIR,
        f'diagnostics_{param}_30day__{RUN_TIMESTAMP}.png'
    )
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')
    print()

    diag_figures.append(fig_path)

log['steps']['step6_diagnostics'] = {
    'node':            PRIMARY_NODE,
    'plots_per_model': ['residuals_over_time', 'residual_histogram'],
    'figures_saved_to': FIGURES_DIR,
    'figures':         diag_figures,
    'note': (
        'Diagnostic only. No regression outputs modified. '
        'Residual pattern or DW far from 2.0 → consider increasing NW lags as robustness check.'
    ),
}

No result for atm_iv — skipping diagnostic plot.
No result for skew — skipping diagnostic plot.
No result for curvature — skipping diagnostic plot.


## Step 7 — Smile parameter time series plots

Plot EU and US smile parameters over time for the primary 30-day node, one panel per parameter (ATM IV, Skew, Curvature).

These plots serve two purposes:
1. **Descriptive statistics**: visual evidence of co-movement (or lack thereof) between EU and US smile parameters; the eyeball test for the spillover hypothesis before any regression
2. **Data quality check**: any discontinuities, outliers, or implausible level shifts visible here should be traced back to the cleaning pipeline (notebooks 03a/03b) before reporting results

**What to look for:**
- EU and US series moving together in level and direction → supports the spillover prior
- One series leading the other visually → directional evidence consistent with $\hat{\beta}_1 > 0$
- Large divergences → may reflect the non-overlapping sample problem (validation mode); note explicitly in thesis if present

In [23]:
# Step 7 — Smile parameter time series plots (30-day primary node)
# One subplot per parameter: EU (t+1) vs US (t) over the sample period.
# Output goes directly into descriptive statistics section.

param_labels = {
    'atm_iv':    'ATM Implied Volatility',
    'skew':      'Skew  [IV(put Δ25) − IV(call Δ75)]',
    'curvature': 'Curvature  [IV(put Δ25) + IV(call Δ75) − 2×ATM IV]'
}

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    f'EU vs US Smile Parameters — {PRIMARY_NODE}-day maturity node',
    fontsize=12, fontweight='bold'
)

for i, param in enumerate(PARAMETERS):
    ax     = axes[i]
    eu_col = f'eu_{param}'
    us_col = f'us_{param}'

    # EU series: date_eu is the t+1 date (EU observation date)
    if eu_col in reg_primary.columns:
        ax.plot(
            reg_primary['date_eu'],
            reg_primary[eu_col],
            label='EU  (date = t+1)',
            color='steelblue',
            linewidth=1.2
        )

    # US series: observed on t, plotted against t+1 date for visual alignment
    # This is intentional — both series share the x-axis date_eu so the
    # one-day lead-lag relationship is visible directly in the plot.
    if us_col in reg_primary.columns:
        ax.plot(
            reg_primary['date_eu'],
            reg_primary[us_col],
            label='US  (date = t, plotted at t+1 for alignment)',
            color='darkorange',
            linewidth=1.2,
            linestyle='--'
        )

    ax.set_ylabel(param_labels.get(param, param), fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[-1].set_xlabel('Date (EU observation date)', fontsize=9)
plt.xticks(rotation=30)
plt.tight_layout()

fig_path = os.path.join(
    FIGURES_DIR,
    f'smile_params_timeseries__{RUN_TIMESTAMP}.png'
)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

log['steps']['step7_timeseries_plots'] = {
    'node':        PRIMARY_NODE,
    'figure':      fig_path,
    'eu_series':   [f'eu_{p}' for p in PARAMETERS],
    'us_series':   [f'us_{p}' for p in PARAMETERS],
    'x_axis':      'date_eu (EU observation date = t+1)',
    'note': (
        'US series plotted at t+1 date for visual alignment with EU series. '
        'The actual US observation is on day t (one trading day prior).'
    ),
}

Saved: ../data/outputs/figures/smile_params_timeseries__20260314_160035.png


/var/folders/yn/ckwt_v6d45v5049z1mf7t11h0000gn/T/ipykernel_13527/3536480629.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Step 8 — Save results table and log

In [24]:
# Build a flat results DataFrame for easy export
rows = []

for param, summ in primary_summaries.items():
    if summ is None:
        continue
    for var, stats in summ['coefficients'].items():
        rows.append({
            'node':        PRIMARY_NODE,
            'model':       param,
            'variable':    var,
            'coef':        stats['coef'],
            'se_nw':       stats['se_nw'],
            'tstat':       stats['tstat'],
            'pvalue':      stats['pvalue'],
            'r_squared':   summ['r_squared'],
            'adj_r_squared': summ['adj_r_squared'],
            'T':           summ['T'],
            'nw_lags':     summ['nw_lags'],
            'durbin_watson': summ['durbin_watson'],
            'specification': 'primary'
        })

for node_str, node_summs in robustness_log.items():
    for param, summ in node_summs.items():
        if summ is None:
            continue
        for var, stats in summ['coefficients'].items():
            rows.append({
                'node':        int(node_str),
                'model':       param,
                'variable':    var,
                'coef':        stats['coef'],
                'se_nw':       stats['se_nw'],
                'tstat':       stats['tstat'],
                'pvalue':      stats['pvalue'],
                'r_squared':   summ['r_squared'],
                'adj_r_squared': summ['adj_r_squared'],
                'T':           summ['T'],
                'nw_lags':     summ['nw_lags'],
                'durbin_watson': summ['durbin_watson'],
                'specification': 'robustness'
            })

results_df = pd.DataFrame(rows)

# Save
results_path = os.path.join(OUTPUT_DIR, f'regression_results__{RUN_TIMESTAMP}.csv')
log_path     = os.path.join(LOG_DIR, f'05_regression_log__{RUN_TIMESTAMP}.json')

results_df.to_csv(results_path, index=False)
print(f'Results table saved: {results_path}')
print(f'Rows: {len(results_df)}')
print()
print(results_df.to_string())

log['output_file'] = results_path
log['status'] = 'complete'

with open(log_path, 'w') as f:
    json.dump(log, f, indent=2, default=str)
print()
print(f'Log saved: {log_path}')

Results table saved: ../data/outputs/regression_results__20260314_160035.csv
Rows: 0

Empty DataFrame
Columns: []
Index: []

Log saved: ../logs/05_regression_log__20260314_160035.json


## Step 9 — Interpretation guide


### What to report for each model

For each of the three models (ATM IV, Skew, Curvature) at the primary node and each robustness node, report:

- $\hat{\beta}_1$ (coefficient on `us_<param>`), its Newey-West SE, t-statistic, and p-value — this is the main result
- $\hat{\beta}_2$ (coefficient on `delta_vix`) and $\hat{\beta}_3$ (coefficient on `eu_<param>_lag1`) — report but these are controls, not the focus
- $R^2$ and adjusted $R^2$
- Sample size $T$ and number of Newey-West lags $n = \lfloor 4(T/100)^{2/9} \rfloor$ used
- Durbin-Watson statistic: near 2.0 = no remaining residual autocorrelation; below 1.5 or above 2.5 = flag

---

### How to interpret $\hat{\beta}_1$

| Outcome | Language to use |
|---|---|
| $\hat{\beta}_1 > 0$, $p < 0.05$ | "...is consistent with the spillover hypothesis" / "suggests that the US smile parameter on day $t$ predicts the EU smile parameter on day $t+1$" |
| $\hat{\beta}_1 > 0$, $0.05 \leq p \leq 0.10$ | "...provides weak evidence of a positive spillover, though statistical significance is marginal" |
| $\hat{\beta}_1 \approx 0$ or $p > 0.10$ | "...no statistically detectable spillover is found for this parameter and maturity combination" |
| $\hat{\beta}_1 < 0$ | Report as-is; note the sign is inconsistent with the directional prior and discuss possible explanations |

**Never use:** "proves", "confirms", "demonstrates causality".  
**Always use:** "is consistent with", "suggests", "estimated association", "we cannot rule out".

---

### Robustness interpretation

If $\hat{\beta}_1$ is significant at the 30-day node but not at other nodes: conclude that the result is maturity-specific and state this as a limitation.  
If $\hat{\beta}_1$ is consistent in sign and significance across nodes: this strengthens — but still does not prove — the spillover interpretation.  
If results flip sign across nodes: treat the primary result with caution and report the instability explicitly.

---

### Limitations to state

**1. Single-stock data, not index options.**  
Results are based on Adidas (EU proxy) and Apple (US proxy) equity options, not SPX or EURO STOXX 50 index options as in Chen et al. (2022). Single-stock implied volatility is noisier and more idiosyncratic. Generalisability to the index-level spillover literature is limited and must be stated explicitly.

**2. Short and non-overlapping sample.**  
The EU and US option data do not share a common calendar window. Regressions run in validation mode (EU-only autocorrelation) do not constitute cross-market spillover evidence. Even when overlap is achieved, a short T produces wide confidence intervals. Coefficient estimates should be treated as indicative, not conclusive.

**3. Linearity assumption.**  
OLS assumes the spillover relationship is linear and constant across the sample. Non-linear dynamics (threshold effects during high-volatility regimes, asymmetric transmission) are not captured. This is a standard limitation of the regression approach in this literature.

**4. No causal identification.**  
The one-day lag structure is motivated by the US market closing before the EU market opens the following day (time-zone identification). However, omitted variables — overnight macro announcements, common global shocks, earnings releases — may confound the estimate. The result is an estimated predictive association, not a causal effect.

**5. Newey-West lag length is a simplified rule.**  
The formula $n = \lfloor 4(T/100)^{2/9} \rfloor$ is passed directly as the HAC bandwidth rather than implementing the full two-step data-dependent procedure of Newey & West (1994). This is standard in applied work but introduces a degree of arbitrariness in the SE correction. Robustness to alternative lag lengths (e.g., $\lfloor 1.5n \rfloor$, $\lfloor 2n \rfloor$) should be noted as a possible extension.